# Understanding Training Basics with PyTorch

**Goal:** understand what training changes, predict the effect of a configuration,
and check that prediction with a small experiment before working with language models.

Our model is `prediction = weight × input + bias`. The four examples follow
`target = 3 × input + 2`, so the perfect fit is **weight 3, bias 2**.

### How to use this notebook

Run the lesson cells from top to bottom. The installation cell is optional if PyTorch
already imports successfully. Individual experiments explicitly reset their parameters;
sections 4–6 first build a prediction and loss, then calculate gradients from them.
Changing a cell or running cells out of order can leave later cells using older values.
Before each experiment, write down your prediction, then compare it with the output.

### Vocabulary

| Term | Meaning in this notebook |
|---|---|
| Parameter | A number adjusted by training: the weight or the bias |
| Weight | The parameter multiplied by the input |
| Bias | The parameter added after multiplication |
| Hyperparameter | A setting we choose, such as learning rate or batch size |
| Prediction | The answer produced by the current parameters |
| Loss | A number measuring prediction error; here we use mean squared error |
| Gradient | How a small change in a parameter affects the current loss |
| Batch | Examples combined to calculate gradients before one update |
| Step | One update of the model's parameters |
| Epoch | One complete pass through the training examples |

**Training loop:** predict → calculate loss → calculate gradients → update parameters → repeat.

### Course organization

Keep training fundamentals in this notebook so related experiments stay together.
Start a separate notebook when the course moves to a new major topic, such as neural
networks or language models. Each experiment should have a heading, description,
commented code, and a short prediction → observation → explanation record.

**Current lesson:** [27. Select and load the best validation checkpoint](#lesson-27).
Run sections 2 and 3, then section 26, then section 27. Write your experiment notes in section 27.

### Contents

- [1. Install PyTorch (optional)](#lesson-1)
- [2. Import PyTorch](#lesson-2)
- [3. Create four training examples](#lesson-3)
- [4. Make predictions with weight and bias](#lesson-4)
- [5. Measure prediction error with mean squared loss](#lesson-5)
- [6. Calculate gradients without changing parameters](#lesson-6)
- [7. Apply one parameter update](#lesson-7)
- [8. Compare learning rates from the same starting point](#lesson-8)
- [9. Recalculate gradients after an update](#lesson-9)
- [10. Repeat the full-batch training loop for 10 steps](#lesson-10)
- [11. Check what happens over 100 steps](#lesson-11)
- [12. Inspect one example at a time: batch size 1](#lesson-12)
- [13. Reverse example order with batch size 1](#lesson-13)
- [14. Train with mini-batches of two examples](#lesson-14)
- [15. Compare batch sizes after one epoch](#lesson-15)
- [16. Compare epochs, updates, and examples processed](#lesson-16)
- [17. Training loss and validation loss](#lesson-17)

- [18. Track training and validation loss over epochs](#lesson-18)

- [19. Fit the supplied targets: original versus noisy data](#lesson-19)

- [20. Early stopping and patience](#lesson-20)

- [21. Evaluate selected checkpoints on a held-out test set](#lesson-21)

- [22. Replace manual updates with the SGD optimizer](#lesson-22)

- [23. Train for 10 steps using SGD](#lesson-23)

- [24. Inspect how momentum changes SGD updates](#lesson-24)

- [25. Compare momentum settings at a fixed training budget](#lesson-25)

- [26. Learning rate and momentum interact](#lesson-26)

- [27. Select and load the best validation checkpoint](#lesson-27)


## 1. Install PyTorch (optional)

<a id="lesson-1"></a>


Use this cell only when setting up the notebook environment. `%pip` installs into
the notebook kernel's Python environment. Restart the kernel if requested after installation.
This cell does not train a model; skip it if the next cell imports PyTorch successfully.


In [5]:
# Optional setup: these commands install packages, not model parameters.
%pip install --upgrade pip
%pip install torch


Defaulting to user installation because normal site-packages is not writeable
  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 2. Import PyTorch

<a id="lesson-2"></a>


Import PyTorch and print the installed version. A tensor is PyTorch's container
for numbers. We will use tensors for both data and adjustable parameters.


In [1]:
import torch

# Confirm that the notebook kernel can use PyTorch.
print("PyTorch version:", torch.__version__)


PyTorch version: 2.6.0+cu124


## 3. Create four training examples

<a id="lesson-3"></a>


`x` holds the inputs and `y` holds the matching correct answers in the same order.
The decimal points create floating-point tensors, which we use for gradient-based training.
`torch.Size([4])` means the tensor is a one-dimensional sequence containing four values.


In [2]:
# Each position in x matches the same position in y.
x = torch.tensor([0., 1., 2., 3.])
y = torch.tensor([2., 5., 8., 11.])

print("Inputs:", x)
print("Correct answers:", y)
print("Shape:", x.shape)


Inputs: tensor([0., 1., 2., 3.])
Correct answers: tensor([ 2.,  5.,  8., 11.])
Shape: torch.Size([4])


## 4. Make predictions with weight and bias

<a id="lesson-4"></a>


The model has **two parameters**: one weight and one bias. Both are shared across all
four examples. `requires_grad=True` asks PyTorch to track calculations involving these
parameters so it can calculate gradients later; it does not train them yet.

**Predict first:** with weight `1` and bias `0`, the predictions should be `[0, 1, 2, 3]`.
`detach()` is used only for displaying values without their gradient-tracking connection.


In [3]:
# Reset the model to our starting parameters.
weight = torch.tensor(1., requires_grad=True)
bias = torch.tensor(0., requires_grad=True)

# The same weight and bias are applied to every input.
predictions = weight * x + bias
print("Predictions:", predictions.detach().tolist())


Predictions: [0.0, 1.0, 2.0, 3.0]


## 5. Measure prediction error with mean squared loss

<a id="lesson-5"></a>


Subtract each correct answer from its prediction, square the error, then average
the four squared errors. Squaring makes both positive and negative errors contribute
positively to loss. A zero loss here means every prediction matches its target.

At the starting parameters, squared errors are `[4, 16, 36, 64]` and their mean is `30`.
Calculating loss alone does not update parameters.


In [4]:
# Compute one squared error for each example.
squared_errors = (predictions - y) ** 2

# Combine the four errors into one mean loss.
loss = squared_errors.mean()
print("Squared errors:", squared_errors.detach().tolist())
print("Mean loss:", loss.item())


Squared errors: [4.0, 16.0, 36.0, 64.0]
Mean loss: 30.0


## 6. Calculate gradients without changing parameters

<a id="lesson-6"></a>


`loss.backward()` calculates gradients and stores them in `.grad`. It does **not**
change the weight or bias. PyTorch accumulates gradients, so clear old gradients before
calculating fresh ones. Assigning `.grad = None` preserves the learned parameter values.

At weight `1`, bias `0`, the gradients are **weight −20, bias −10**. A negative gradient
means increasing that parameter would reduce loss locally. A sufficiently large update
can still overshoot. Rebuilding predictions and loss creates a fresh calculation graph.


In [5]:
# Clear stored gradients; the learned weight and bias stay unchanged.
weight.grad = None
bias.grad = None

# Recalculate the loss from the current parameters using a fresh graph.
predictions = weight * x + bias
loss = ((predictions - y) ** 2).mean()

# Calculate gradients, without performing an update.
loss.backward()
print("Weight gradient:", weight.grad.item())
print("Bias gradient:", bias.grad.item())


Weight gradient: -20.0
Bias gradient: -10.0


## 7. Apply one parameter update

<a id="lesson-7"></a>


The update rule is **new parameter = old parameter − learning rate × gradient**.
The learning rate scales the update; it does not change the gradient calculated at a
fixed set of parameters. Subtracting a negative gradient increases the parameter.

This experiment resets to weight `1`, bias `0` and recalculates gradients so rerunning
this cell always demonstrates exactly one update. With learning rate `0.05`, expect
weight `2`, bias `0.5`, and loss to fall from `30` to `10.25`.

`torch.no_grad()` stops PyTorch from tracking the update itself. The parameters still change.


In [6]:
# Reset so repeated runs of this cell start from the same position.
weight = torch.tensor(1., requires_grad=True)
bias = torch.tensor(0., requires_grad=True)
learning_rate = 0.05

# Calculate fresh gradients at this starting point.
predictions = weight * x + bias
loss = ((predictions - y) ** 2).mean()
loss.backward()
before = loss.item()

with torch.no_grad():
    # Apply the gradients to both parameters without tracking the update.
    weight -= learning_rate * weight.grad
    bias -= learning_rate * bias.grad

    # Measure predictions and loss again AFTER the update.
    new_predictions = weight * x + bias
    after = ((new_predictions - y) ** 2).mean().item()

print("New weight:", weight.item())
print("New bias:", bias.item())
print("New predictions:", new_predictions.tolist())
print("Loss before:", before)
print("Loss after:", after)


New weight: 2.0
New bias: 0.5
New predictions: [0.5, 2.5, 4.5, 6.5]
Loss before: 30.0
Loss after: 10.25


## 8. Compare learning rates from the same starting point

<a id="lesson-8"></a>


Change only the learning rate: every run starts at weight `1`, bias `0`, uses all four
examples, and makes one update. The starting gradients are identical across these runs.

**Predict first:** which rate will improve loss most after one update? Rate `0.01`
improves slowly, `0.05` improves more, and `0.50` overshoots, raising loss to `305`.
The conclusion is that step size matters; the smallest rate is not always the best choice.
The function uses local `w` and `b`, so it does not change the earlier `weight` and `bias`.


In [7]:
def one_step(lr):
    """Reset a model, apply one full-batch update, and return its results."""
    # A fresh start makes learning-rate comparisons reproducible.
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)

    initial_loss = ((w * x + b - y) ** 2).mean()
    initial_loss.backward()

    with torch.no_grad():
        # lr scales the gradients; it does not calculate them.
        w -= lr * w.grad
        b -= lr * b.grad
        final_loss = ((w * x + b - y) ** 2).mean()

    return w.item(), b.item(), initial_loss.item(), final_loss.item()

print("rate   weight   bias   loss before   loss after")
for rate in [0.01, 0.05, 0.5]:
    w, b, old, new = one_step(rate)
    print(f"{rate:<6.2f} {w:<8.2f} {b:<6.2f} {old:<13.2f} {new:.2f}")


rate   weight   bias   loss before   loss after
0.01   1.20     0.10   30.00         25.21
0.05   2.00     0.50   30.00         10.25
0.50   11.00    5.00   30.00         305.00


## 9. Recalculate gradients after an update

<a id="lesson-9"></a>


An update changes predictions and errors, so the next step needs new gradients.
This cell starts at the first update's result, weight `2`, bias `0.5`, rather than
reusing the initial gradients −20 and −10. Expect **weight gradient −11.5, bias gradient −6**.
With learning rate `0.05`, the next weight would be `2.575`, not `2.525`.
This cell calculates the new gradients without applying another update.


In [8]:
# Recreate the first update's result to make this demonstration repeatable.
weight = torch.tensor(2., requires_grad=True)
bias = torch.tensor(0.5, requires_grad=True)

# Clear gradients before calculating them at the new parameter values.
weight.grad = None
bias.grad = None
predictions = weight * x + bias
loss = ((predictions - y) ** 2).mean()
loss.backward()

print("Weight gradient:", weight.grad.item())
print("Bias gradient:", bias.grad.item())


Weight gradient: -11.5
Bias gradient: -6.0


## 10. Repeat the full-batch training loop for 10 steps

<a id="lesson-10"></a>


Reset the parameters, then repeat predict → loss → gradients → update ten times.
Each printed row reports loss **after** the update. All four examples contribute to
each update, so batch size is `4`. One step is also one epoch in this particular loop.

The learning rate stays fixed, but each update uses new gradients. Expect loss to
decrease by different amounts, reaching about `0.1513`. Weight can exceed `3` while
bias remains below `2`: the two parameters jointly reduce prediction errors.


In [9]:
# Reset once, before training; preserve learned values between loop iterations.
weight = torch.tensor(1., requires_grad=True)
bias = torch.tensor(0., requires_grad=True)
learning_rate = 0.05

for step in range(10):
    # Clear previous gradients, not the learned parameters.
    weight.grad = None
    bias.grad = None

    # Use all four examples to calculate one mean loss.
    predictions = weight * x + bias
    loss = ((predictions - y) ** 2).mean()
    loss.backward()

    with torch.no_grad():
        # One update per iteration, after combining all four errors.
        weight -= learning_rate * weight.grad
        bias -= learning_rate * bias.grad
        updated_loss = ((weight * x + bias - y) ** 2).mean()

    print(
        f"Step {step + 1:2d} | "
        f"weight={weight.item():.4f} | "
        f"bias={bias.item():.4f} | "
        f"loss={updated_loss.item():.4f}"
    )


Step  1 | weight=2.0000 | bias=0.5000 | loss=10.2500
Step  2 | weight=2.5750 | bias=0.8000 | loss=3.6022
Step  3 | weight=2.9037 | bias=0.9837 | loss=1.3586
Step  4 | weight=3.0899 | bias=1.0998 | loss=0.5959
Step  5 | weight=3.1934 | bias=1.1763 | loss=0.3314
Step  6 | weight=3.2493 | bias=1.2297 | loss=0.2348
Step  7 | weight=3.2776 | bias=1.2693 | loss=0.1951
Step  8 | weight=3.2900 | bias=1.3008 | loss=0.1749
Step  9 | weight=3.2934 | bias=1.3272 | loss=0.1618
Step 10 | weight=3.2916 | bias=1.3505 | loss=0.1513


## 11. Check what happens over 100 steps

<a id="lesson-11"></a>


**Our hypothesis:** loss might reach its minimum and then increase if training keeps
moving past the solution. Test that hypothesis by resetting and running 100 steps.
Here, at learning rate `0.05`, loss decreases at every step; weight approaches `3`
and bias approaches `2`.

A fixed learning rate does not imply a fixed update. Smaller gradients produce smaller
changes. If this model's squared-error loss is exactly zero, its gradients are zero,
so these plain gradient-descent updates stop. Too large a learning rate can still overshoot.
These are training-loss results on the same four examples; they do not measure generalization.


In [10]:
# Reset for a separate experiment with a longer training duration.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
learning_rate = 0.05
with torch.no_grad():
    previous_loss = ((w * x + b - y) ** 2).mean().item()
increases = []

for step in range(1, 101):
    w.grad = None
    b.grad = None
    loss = ((w * x + b - y) ** 2).mean()
    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        updated_loss = ((w * x + b - y) ** 2).mean().item()

    # Check every step, even though we print only selected checkpoints.
    if updated_loss > previous_loss:
        increases.append(step)
    previous_loss = updated_loss

    if step in (10, 20, 50, 100):
        print(
            f"Step {step:3d} | weight={w.item():.4f} | "
            f"bias={b.item():.4f} | loss={updated_loss:.8f}"
        )

print("Steps where loss increased:", increases)


Step  10 | weight=3.2916 | bias=1.3505 | loss=0.15129718
Step  20 | weight=3.2232 | bias=1.5233 | loss=0.08241145
Step  50 | weight=3.0902 | bias=1.8073 | loss=0.01346400
Step 100 | weight=3.0199 | bias=1.9574 | loss=0.00065739
Steps where loss increased: []


## 12. Inspect one example at a time: batch size 1

<a id="lesson-12"></a>


Reset and process inputs `0`, `1`, `2`, `3` in that order. Each example gets its own
loss, gradients, and update. The next example uses the newly learned parameters.
Four updates complete **one epoch**, because each of our four examples has been used once.

- At `x = 0`, prediction equals bias. Weight has no effect, so its gradient is zero.
- At `x = 1`, changing weight or bias by the same amount has the same effect on prediction;
  their gradients are equal.
- At `x = 2`, weight's gradient has twice the magnitude of bias's gradient.

Print gradients **before** the update and parameters **after** it. Measure mean loss
over the entire dataset at the end, so the reported loss covers all four examples.
Long decimals and `-0.0` are normal floating-point representations; `-0.0` is zero.


In [11]:
# Reset before the epoch; keep learned parameters between examples.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
learning_rate = 0.05

for i in range(len(x)):
    # The previous example's gradients must not carry into this example.
    w.grad = None
    b.grad = None

    # A batch of one contains one prediction and one squared error.
    prediction = w * x[i] + b
    loss = (prediction - y[i]) ** 2
    loss.backward()

    print(
        f"Input {x[i].item():.0f} | prediction={prediction.item():.4f} | "
        f"weight gradient={w.grad.item():.4f} | bias gradient={b.grad.item():.4f}"
    )

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    print(f"Step {i + 1} | weight={w.item():.4f} | bias={b.item():.4f}")

with torch.no_grad():
    # Evaluate the final parameters on ALL examples, not just the last one.
    total_loss = ((w * x + b - y) ** 2).mean()

forward_result = (w.item(), b.item(), total_loss.item())
print("Loss across all examples:", total_loss.item())


Input 0 | prediction=0.0000 | weight gradient=-0.0000 | bias gradient=-4.0000
Step 1 | weight=1.0000 | bias=0.2000
Input 1 | prediction=1.2000 | weight gradient=-7.6000 | bias gradient=-7.6000
Step 2 | weight=1.3800 | bias=0.5800
Input 2 | prediction=3.3400 | weight gradient=-18.6400 | bias gradient=-9.3200
Step 3 | weight=2.3120 | bias=1.0460
Input 3 | prediction=7.9820 | weight gradient=-18.1080 | bias gradient=-6.0360
Step 4 | weight=3.2174 | bias=1.3478
Loss across all examples: 0.16541965305805206


## 13. Reverse example order with batch size 1

<a id="lesson-13"></a>


**Predict first:** will reversing the order change the final parameters? Keep the same
starting parameters and learning rate, and process inputs `3`, `2`, `1`, `0`.
Order matters because each update changes the parameters used for the next example.
Expect a final loss near `0.4012`, compared with `0.1654` in forward order.

With all examples in one batch, rearranging input-target pairs does not mathematically
change their mean loss or combined gradients at fixed parameters. Small numerical rounding
differences can occur. Keep each input paired with its own target when rearranging data.


In [12]:
# Reset so order is the only changed setting.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
learning_rate = 0.05

# Iterate over positions 3, 2, 1, 0, keeping each x paired with its matching y.
for step, i in enumerate(range(len(x) - 1, -1, -1), start=1):
    w.grad = None
    b.grad = None
    loss = (w * x[i] + b - y[i]) ** 2
    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    print(
        f"Step {step} | input={x[i].item():.0f} | "
        f"weight={w.item():.4f} | bias={b.item():.4f}"
    )

with torch.no_grad():
    total_loss = ((w * x + b - y) ** 2).mean()

print("Loss across all examples:", total_loss.item())


Step 1 | input=3 | weight=3.4000 | bias=0.8000
Step 2 | input=2 | weight=3.4800 | bias=0.8400
Step 3 | input=1 | weight=3.5480 | bias=0.9080
Step 4 | input=0 | weight=3.5480 | bias=1.0172
Loss across all examples: 0.40123653411865234


## 14. Train with mini-batches of two examples

<a id="lesson-14"></a>


Reset and group inputs into `[0, 1]` and `[2, 3]`. Both examples in a batch use the same
current parameters. Average their squared errors, calculate gradients, and update **once**.
The second batch then uses the updated parameters. Two updates complete one epoch.

`range(0, len(x), 2)` gives starting positions `0` and `2`; `x[start:start + 2]`
selects the two consecutive inputs. `.mean()` averages errors; it does not sum them.
Expect final weight about `2.795`, bias `0.920`, and full-dataset loss `1.9777`.


In [13]:
# Reset to compare with the other batch sizes after one epoch.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
learning_rate = 0.05

for start in range(0, len(x), 2):
    # Select two inputs and their corresponding targets.
    batch_x = x[start:start + 2]
    batch_y = y[start:start + 2]

    w.grad = None
    b.grad = None

    # Combine both examples BEFORE calculating gradients and updating.
    predictions = w * batch_x + b
    loss = ((predictions - batch_y) ** 2).mean()
    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    print(
        f"Inputs: {batch_x.tolist()} | "
        f"weight={w.item():.4f} | bias={b.item():.4f}"
    )

with torch.no_grad():
    # Use the same full-dataset metric as the batch-size-1 experiment.
    total_loss = ((w * x + b - y) ** 2).mean()

mini_batch_result = (w.item(), b.item(), total_loss.item())
print("Loss across all examples:", total_loss.item())


Inputs: [0.0, 1.0] | weight=1.2000 | bias=0.3000
Inputs: [2.0, 3.0] | weight=2.7950 | bias=0.9200
Loss across all examples: 1.977686882019043


## 15. Compare batch sizes after one epoch

<a id="lesson-15"></a>


Compare the saved forward-order results for batch sizes `1` and `2` with a fresh single
full-batch update. All runs start at weight `1`, bias `0`, with learning rate `0.05`.
Each processes four examples, but the update counts differ.

Smaller batches made more progress **in this experiment**, but also made more updates.
This does not establish that smaller batches always perform better. Batch size affects
both the examples contributing to each gradient and the frequency of parameter updates.
Run sections 8, 12, and 14 before this comparison cell so its function and saved results exist.


In [14]:
# Reuse one_step to get the full-batch result from the same initial parameters.
full_w, full_b, initial_loss, full_loss = one_step(0.05)

# Each row represents one epoch: each of the four examples is used once.
print("Batch size | Updates | Weight | Bias   | Full-dataset loss")
for batch_size, updates, result in [
    (4, 1, (full_w, full_b, full_loss)),
    (2, 2, mini_batch_result),
    (1, 4, forward_result),
]:
    final_w, final_b, final_loss = result
    print(
        f"{batch_size:10d} | {updates:7d} | "
        f"{final_w:.4f} | {final_b:.4f} | {final_loss:.4f}"
    )


Batch size | Updates | Weight | Bias   | Full-dataset loss
         4 |       1 | 2.0000 | 0.5000 | 10.2500
         2 |       2 | 2.7950 | 0.9200 | 1.9777
         1 |       4 | 3.2174 | 1.3478 | 0.1654


## 16. Compare epochs, updates, and examples processed

<a id="lesson-16"></a>

For this four-example dataset:

| Batch size | Updates per epoch | Epochs needed for four updates | Examples processed over those epochs |
|---|---:|---:|---:|
| 1 | 4 | 1 | 4 |
| 2 | 2 | 2 | 8 |
| 4 | 1 | 4 | 16 |

Batch size `4` needs **four epochs** to match the four updates made by batch size `1`
in one epoch. But it processes sixteen examples in total: the same four examples
revisited four times. Equal epochs, equal updates, and equal examples processed are
different ways to compare training runs. State which budget you are holding fixed.

### What we learned

- Weight is a kind of parameter; bias is another parameter.
- Loss measures error; gradients describe how parameter changes affect that loss.
- `backward()` calculates gradients; the update code changes parameters.
- Clearing `.grad` clears stored gradients while preserving learned parameters.
- The learning rate scales gradients into updates. A fixed rate does not imply fixed-sized updates.
- A batch contributes to one update. An epoch covers the whole dataset once.
- With batch size 1, each update changes the starting point for the next example, so order matters.
- Every loss compared in sections 1–16 is measured on the same four training examples.
  Section 17 evaluates unseen examples; overfitting is a later lesson.


## 17. Training loss and validation loss

<a id="lesson-17"></a>

Training loss measures error on examples used for parameter updates. Validation loss
measures error on reserved examples that did not contribute to those updates.
It helps us check whether the learned pattern works on unseen inputs.

**Run section 14 immediately before this cell.** It trains `w` and `b` for one epoch
with batch size 2, ending at approximately weight `2.795`, bias `0.920`.
The training inputs are `[0, 1, 2, 3]`; the validation inputs are `[4, 5]` with targets
`[14, 17]`. Both sets follow `y = 3 × x + 2`, but the validation inputs extend beyond
the training input range.

Use the same trained parameters for both evaluations. This cell only predicts and
measures loss: it has no `backward()` call and no parameter updates. `torch.no_grad()`
also avoids tracking the evaluation calculations for gradients.

**Predict first:** will training and validation loss be identical? Record your reason
before running. A higher validation loss in this single measurement does not establish
overfitting; we have not yet measured how both losses change across training.

### Experiment notes

- **Configuration:** one epoch, batch size 2, learning rate 0.05; forward example order.
- **Prediction before running:** _Write your prediction here._
- **Observed results:** _Record both losses and the validation predictions._
- **Explanation:** _What does the result tell you about the learned parameters?_


In [ ]:
# Reserved examples: these did not contribute to training updates.
validation_x = torch.tensor([4., 5.])
validation_y = torch.tensor([14., 17.])

with torch.no_grad():
    # Measure performance on the examples used for training.
    training_predictions = w * x + b
    training_loss = ((training_predictions - y) ** 2).mean()

    # Use the SAME parameters on unseen examples, without updating them.
    validation_predictions = w * validation_x + b
    validation_loss = (
        (validation_predictions - validation_y) ** 2
    ).mean()

print("Weight:", w.item())
print("Bias:", b.item())
print("Validation predictions:", validation_predictions.tolist())
print("Training loss:", training_loss.item())
print("Validation loss:", validation_loss.item())


## 18. Track training and validation loss over epochs

<a id="lesson-18"></a>

**Prediction before running:** training loss will decrease while validation loss increases.

Test this prediction with the same four training examples, two reserved validation
examples, batch size 2, learning rate 0.05, and forward example order. The changed
setting is training duration: we train for 100 epochs and evaluate after each epoch.
Run section 3 first to create `x` and `y`; this cell resets its own parameters and
validation examples. Its first epoch should reproduce sections 14 and 17.

Reset parameters only once, before the outer epoch loop. Each epoch has two batches
and therefore two updates. Across 100 epochs, this experiment makes 200 updates.
The validation examples never contribute to gradients or parameter updates.

A higher validation loss at one point does not by itself establish overfitting.
Inspect how both losses change across epochs. Training can improve performance on
unseen examples; overfitting is a possible outcome, not an automatic consequence of
training for more epochs. These validation inputs follow the same noiseless linear
rule as the training inputs.

### Experiment notes

- **Prediction:** training loss falls while validation loss rises.
- **Observation:** _Record how both losses change between epochs 1, 10, and 100._
- **Explanation:** _Did the result support the prediction? How close are the final parameters to weight 3 and bias 2?_


In [ ]:
# Reset once before training, not at the beginning of every epoch.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
learning_rate = 0.05
batch_size = 2

# Reserve these examples for evaluation only.
validation_x = torch.tensor([4., 5.])
validation_y = torch.tensor([14., 17.])
history = []

for epoch in range(1, 101):
    # One epoch visits all four training examples in two batches.
    for start in range(0, len(x), batch_size):
        batch_x = x[start:start + batch_size]
        batch_y = y[start:start + batch_size]

        w.grad = None
        b.grad = None
        predictions = w * batch_x + b
        loss = ((predictions - batch_y) ** 2).mean()
        loss.backward()

        with torch.no_grad():
            w -= learning_rate * w.grad
            b -= learning_rate * b.grad

    # Evaluate both datasets at the SAME parameter values after this epoch.
    # This block makes predictions and measures loss without updating anything.
    with torch.no_grad():
        training_loss = ((w * x + b - y) ** 2).mean().item()
        validation_loss = (
            (w * validation_x + b - validation_y) ** 2
        ).mean().item()

    # Save every epoch so we can inspect the complete learning history later.
    history.append((epoch, training_loss, validation_loss))

    # Print checkpoints to keep the notebook output short and readable.
    if epoch in (1, 2, 5, 10, 20, 50, 100):
        print(
            f"Epoch {epoch:3d} | training loss={training_loss:.6f} | "
            f"validation loss={validation_loss:.6f}"
        )

print(f"Final weight: {w.item():.6f} | Final bias: {b.item():.6f}")


## 19. Fit the supplied targets: original versus noisy data

<a id="lesson-19"></a>

**Question:** can fitting the supplied training answers more closely make performance
on unseen examples worse? The loss uses the supplied target; it cannot know whether
we supplied a wrong answer.

Make an independent copy of `y` using `clone()`, then change the answer for input 3
from 11 to 14. The original `y` stays unchanged. The reserved validation examples
continue to follow the original rule `y = 3 × x + 2`.

Compare two runs with identical settings: starting weight 1, bias 0, learning rate
0.05, batch size 2, forward order, and 100 epochs. **Only the training targets differ.**
Run sections 2 and 3 first to import PyTorch and create `x` and `y`.

Each run's training loss is measured against its own training targets. Both validation
losses use the same correct validation targets. Look at the trend across epochs, rather
than assuming every metric must decrease at every epoch: mini-batch updates follow
individual batch gradients, so full-dataset training loss can fluctuate too.

Whenever validation loss reaches a new low, save a checkpoint: the epoch, validation
loss, weight, and bias. Saving `.item()` values captures the numbers at that moment.
We continue through all 100 epochs so we can see how later training compares with the
best saved checkpoint; this experiment does not stop training early.

### Experiment notes

- **Prediction:** the original targets should support the true pattern; the changed target may pull the model away from it as training continues.
- **Observation:** _Compare each run's validation loss at epochs 1, 3, and 100. Which checkpoint had the lowest validation loss?_
- **Explanation:** _Why can a model improve its fit to incorrect training answers while making worse predictions on correct unseen examples?_


In [ ]:
# Preserve the original targets; modify only an independent copy.
noisy_y = y.clone()
noisy_y[3] = 14.

# These correct targets never contribute to training updates.
validation_x = torch.tensor([4., 5.])
validation_y = torch.tensor([14., 17.])
learning_rate = 0.05
batch_size = 2
experiment_results = {}

for label, training_targets in [
    ("Original targets", y),
    ("Changed target", noisy_y),
]:
    # Reset each run so both start from exactly the same parameters.
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    run_history = []
    best_checkpoint = None
    print(f"\n{label}: {training_targets.tolist()}")

    for epoch in range(1, 101):
        for start in range(0, len(x), batch_size):
            batch_x = x[start:start + batch_size]
            batch_y = training_targets[start:start + batch_size]

            w.grad = None
            b.grad = None
            loss = ((w * batch_x + b - batch_y) ** 2).mean()
            loss.backward()

            with torch.no_grad():
                w -= learning_rate * w.grad
                b -= learning_rate * b.grad

        # Evaluate both losses after the epoch without changing the parameters.
        with torch.no_grad():
            training_loss = ((w * x + b - training_targets) ** 2).mean().item()
            validation_loss = (
                (w * validation_x + b - validation_y) ** 2
            ).mean().item()

        run_history.append((epoch, training_loss, validation_loss))

        # Choose checkpoints by VALIDATION loss, not training loss.
        if best_checkpoint is None or validation_loss < best_checkpoint["validation_loss"]:
            best_checkpoint = {
                "epoch": epoch,
                "validation_loss": validation_loss,
                "weight": w.item(),
                "bias": b.item(),
            }

        if epoch in (1, 2, 3, 5, 10, 20, 50, 100):
            print(
                f"Epoch {epoch:3d} | training loss={training_loss:.6f} | "
                f"validation loss={validation_loss:.6f}"
            )

    # Retain each run's learning history and best checkpoint for later reference.
    experiment_results[label] = {
        "history": run_history,
        "best_checkpoint": best_checkpoint,
    }
    print(f"Final weight: {w.item():.6f} | Final bias: {b.item():.6f}")
    print("Best checkpoint by validation loss:", best_checkpoint)


## 20. Early stopping and patience

<a id="lesson-20"></a>

**Prediction:** patience 2 usually stops sooner than patience 50 because it tolerates
fewer consecutive epochs without a new best validation loss.

Run sections 2 and 3 first to import PyTorch and create the original `x` and `y`.
Compare patience 2 and 50 on both original and changed targets. Every run starts at
weight 1, bias 0, with batch size 2, learning rate 0.05, forward order, and at most
100 epochs. Only the patience setting differs within each dataset comparison.
Patience 50 is an illustrative choice for this experiment, not a universal recommendation.

The early-stopping counter tracks **epochs without a new best validation loss**, not
just epochs where loss rises from the immediately preceding epoch. Any new best resets
the counter to zero. When the counter reaches patience, stop and restore the best
checkpoint. The parameters from the stopping epoch may differ from the best checkpoint.

Stopping early saves updates, but short patience can miss a later recovery. Recall
that the original-target run had a very low validation loss at epoch 2, followed by a
temporary rise, yet eventually performed better at epoch 100.

### Experiment notes

- **Prediction:** _Which dataset might benefit from waiting longer?_
- **Observation:** _For each run, record its stopping epoch, best epoch, and restored validation loss._
- **Explanation:** _Why might short patience help with the changed target but stop the original-target run too soon?_


In [ ]:
def train_with_early_stopping(training_targets, patience, max_epochs=100):
    """Train until patience is exhausted or the epoch limit is reached."""
    # Local parameters give every experiment an independent, identical start.
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    learning_rate = 0.05
    batch_size = 2
    validation_x = torch.tensor([4., 5.])
    validation_y = torch.tensor([14., 17.])

    best_checkpoint = None
    epochs_without_improvement = 0
    history = []
    stop_reason = "Reached epoch limit"

    for epoch in range(1, max_epochs + 1):
        for start in range(0, len(x), batch_size):
            batch_x = x[start:start + batch_size]
            batch_y = training_targets[start:start + batch_size]

            w.grad = None
            b.grad = None
            loss = ((w * batch_x + b - batch_y) ** 2).mean()
            loss.backward()

            with torch.no_grad():
                w -= learning_rate * w.grad
                b -= learning_rate * b.grad

        # Validation only measures performance; it never updates parameters.
        with torch.no_grad():
            training_loss = ((w * x + b - training_targets) ** 2).mean().item()
            validation_loss = (
                (w * validation_x + b - validation_y) ** 2
            ).mean().item()
        history.append((epoch, training_loss, validation_loss))

        if best_checkpoint is None or validation_loss < best_checkpoint["validation_loss"]:
            # Save independent numeric values, not references to changing tensors.
            best_checkpoint = {
                "epoch": epoch,
                "validation_loss": validation_loss,
                "weight": w.item(),
                "bias": b.item(),
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            stop_reason = "Reached patience"
            break

    # Restore the BEST checkpoint, not necessarily the last trained parameters.
    with torch.no_grad():
        w.fill_(best_checkpoint["weight"])
        b.fill_(best_checkpoint["bias"])
        restored_validation_loss = (
            (w * validation_x + b - validation_y) ** 2
        ).mean().item()

    return {
        "stopped_epoch": epoch,
        "stop_reason": stop_reason,
        "best_checkpoint": best_checkpoint,
        "restored_validation_loss": restored_validation_loss,
        "history": history,
    }


# Keep the original data intact and create the same changed target as section 19.
noisy_y = y.clone()
noisy_y[3] = 14.
early_stopping_results = {}

for label, targets in [("Original targets", y), ("Changed target", noisy_y)]:
    for patience in (2, 50):
        result = train_with_early_stopping(targets, patience)
        early_stopping_results[(label, patience)] = result
        best = result["best_checkpoint"]
        print(
            f"{label} | patience={patience:2d} | "
            f"stopped epoch={result['stopped_epoch']:3d} | "
            f"best epoch={best['epoch']:3d} | "
            f"restored validation loss={result['restored_validation_loss']:.6f}"
        )
        print("Stop reason:", result["stop_reason"])


## 21. Evaluate selected checkpoints on a held-out test set

<a id="lesson-21"></a>

| Dataset | Purpose |
|---|---|
| Training | Calculate gradients and update parameters |
| Validation | Choose settings and checkpoints |
| Test | Evaluate the chosen model after those decisions |

Validation examples do not produce parameter updates, but they influence our decisions.
A separate test set provides a final check on examples reserved from both training and
validation-based selection. Do not repeatedly tune settings to improve the test result;
that would make the test set part of the selection process too.

We already selected the original-target checkpoint with patience 50, and the changed-target
checkpoint with patience 2, using section 20's validation results. These choices are fixed
before this evaluation. Run sections 2 and 3, then section 20, before this cell.

Reserve test inputs `[6, 7]` with correct answers `[20, 23]`, following the same rule
`y = 3 × x + 2`. Their inputs extend beyond the training and validation ranges. Evaluate
both selected checkpoints without gradients or parameter updates. The test loss describes
performance on these two examples; a small test set does not establish performance on all
possible inputs.

### Experiment notes

- **Prediction before running:** _How will each selected checkpoint perform on these test inputs?_
- **Observation:** _Record both predictions and each test loss._
- **Explanation:** _Why might a checkpoint with small validation loss still have a larger loss on test inputs farther from the training range?_


In [ ]:
# New examples: reserved from training updates and checkpoint selection.
test_x = torch.tensor([6., 7.])
test_y = torch.tensor([20., 23.])
test_results = {}

# These configurations were chosen using validation loss, before checking test loss.
selected_runs = [("Original targets", 50), ("Changed target", 2)]

for label, patience in selected_runs:
    checkpoint = early_stopping_results[(label, patience)]["best_checkpoint"]

    # Load the saved numeric values, rather than using the last global w and b.
    test_w = torch.tensor(checkpoint["weight"])
    test_b = torch.tensor(checkpoint["bias"])

    with torch.no_grad():
        test_predictions = test_w * test_x + test_b
        test_loss = ((test_predictions - test_y) ** 2).mean().item()

    test_results[label] = {
        "selected_epoch": checkpoint["epoch"],
        "predictions": test_predictions.tolist(),
        "test_loss": test_loss,
    }
    print(f"\n{label} | selected epoch={checkpoint['epoch']}")
    print("Test inputs:", test_x.tolist())
    print("Correct answers:", test_y.tolist())
    print("Test predictions:", test_predictions.tolist())
    print(f"Test loss: {test_loss:.8f}")


## 22. Replace manual updates with the SGD optimizer

<a id="lesson-22"></a>

An optimizer applies parameter updates using calculated gradients. It does not replace
prediction, loss calculation, or `loss.backward()`. Plain SGD, with momentum and weight
decay set to zero, uses the update rule we already learned:

`new parameter = old parameter - learning rate * gradient`

| Manual operation | Optimizer operation |
|---|---|
| Set each parameter's `.grad` to `None` | `optimizer.zero_grad(set_to_none=True)` |
| Calculate gradients with `loss.backward()` | Still `loss.backward()` |
| Subtract learning rate times gradient from each parameter | `optimizer.step()` |

Give the optimizer the parameter tensors it should update and choose a learning rate.
`step()` updates those parameters using their stored gradients. It does not clear the
gradients; call `zero_grad()` before the next gradient calculation. For this basic SGD
configuration, the optimizer handles applying the update without tracking it for gradients.

We compare one manual update with one optimizer update. Both start at weight 1, bias 0,
use all four original examples, and have learning rate 0.05. The SGD name does not force
batch size 1: the examples used to calculate loss determine the batch size.
Run sections 2 and 3 first. **Predict:** both approaches should produce the same weight,
bias, and loss, apart from possible tiny floating-point rounding differences.

### Experiment notes

- **Prediction:** manual updates and plain SGD should agree.
- **Observation:** _Compare their gradients before the update and their final parameters and losses._
- **Explanation:** _What does `step()` replace? Which operation still calculates the gradients?_

API references: [PyTorch SGD documentation](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)
and [zero_grad documentation](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html).
The exercise was also executed successfully with this machine's installed PyTorch.


In [ ]:
learning_rate = 0.05

# Experiment A: the manual update we already understand.
manual_w = torch.tensor(1., requires_grad=True)
manual_b = torch.tensor(0., requires_grad=True)
manual_w.grad = None
manual_b.grad = None
manual_loss = ((manual_w * x + manual_b - y) ** 2).mean()
manual_loss.backward()
print("Manual gradients:", manual_w.grad.item(), manual_b.grad.item())

with torch.no_grad():
    manual_w -= learning_rate * manual_w.grad
    manual_b -= learning_rate * manual_b.grad
    manual_after = ((manual_w * x + manual_b - y) ** 2).mean().item()

# Experiment B: identical parameters and data, using an optimizer for the update.
sgd_w = torch.tensor(1., requires_grad=True)
sgd_b = torch.tensor(0., requires_grad=True)
optimizer = torch.optim.SGD(
    [sgd_w, sgd_b], lr=learning_rate, momentum=0, weight_decay=0
)

# Clear gradients for both parameters registered with this optimizer.
optimizer.zero_grad(set_to_none=True)
sgd_loss = ((sgd_w * x + sgd_b - y) ** 2).mean()
sgd_loss.backward()  # This still calculates the gradients; no parameter update yet.
print("SGD gradients:", sgd_w.grad.item(), sgd_b.grad.item())
print("SGD parameters before step:", sgd_w.item(), sgd_b.item())

optimizer.step()  # Replaces our manual parameter-update lines.

with torch.no_grad():
    sgd_after = ((sgd_w * x + sgd_b - y) ** 2).mean().item()

print(f"Manual | weight={manual_w.item():.4f} | bias={manual_b.item():.4f} | loss={manual_after:.4f}")
print(f"SGD    | weight={sgd_w.item():.4f} | bias={sgd_b.item():.4f} | loss={sgd_after:.4f}")
print("Parameters agree:", torch.allclose(manual_w, sgd_w) and torch.allclose(manual_b, sgd_b))


## 23. Train for 10 steps using SGD

<a id="lesson-23"></a>

Section 22 showed that a single plain SGD update matches our manual update. Now repeat
10 updates, matching section 10's original full-batch training experiment.
Run sections 2 and 3 first. This cell resets the parameters and creates a fresh optimizer.

Create the parameters and optimizer **once before the loop**. Keep the learned parameter
values between iterations. Every step clears old gradients, calculates a fresh loss and
fresh gradients, then calls `step()` to update both registered parameters.

| Operation | Purpose |
|---|---|
| `optimizer.zero_grad(set_to_none=True)` | Clear stored gradients without resetting parameters |
| `predictions = w * x + b` | Predict using the current parameters |
| `loss = ((predictions - y) ** 2).mean()` | Combine all four squared errors into one loss |
| `loss.backward()` | Calculate gradients |
| `optimizer.step()` | Apply one parameter update |

All four examples are in each batch, so each step is one complete epoch. Print loss
**after** each update. `step()` does not clear gradients automatically; the next iteration's
`zero_grad()` handles that. Plain SGD here has no momentum or weight decay.

**Predict before running:** the ten rows should match section 10, apart from possible tiny
rounding differences. The optimizer changes how we express the update, not its mathematical rule.

### Experiment notes

- **Prediction:** plain SGD should follow the same trajectory as the manual loop.
- **Observation:** _Compare the first, second, and tenth updates with section 10._
- **Explanation:** _Why do we clear gradients every step but initialize parameters only once?_


In [ ]:
# Initialize once; do not reset the learned parameters inside the loop.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.05, momentum=0, weight_decay=0)
sgd_history = []

for step in range(1, 11):
    # Clear old gradients while preserving the learned weight and bias.
    optimizer.zero_grad(set_to_none=True)

    # Use all four examples to calculate fresh predictions, loss, and gradients.
    predictions = w * x + b
    loss = ((predictions - y) ** 2).mean()
    loss.backward()

    # Update both parameters registered with this optimizer.
    optimizer.step()

    # Measure loss AFTER the update without building a gradient-tracking graph.
    with torch.no_grad():
        updated_loss = ((w * x + b - y) ** 2).mean().item()

    sgd_history.append((step, w.item(), b.item(), updated_loss))
    print(
        f"Step {step:2d} | weight={w.item():.4f} | "
        f"bias={b.item():.4f} | loss={updated_loss:.4f}"
    )


## 24. Inspect how momentum changes SGD updates

<a id="lesson-24"></a>

Plain SGD uses the current gradient for its parameter update. SGD with momentum also
keeps a running combination of earlier gradients. For the settings used here:

`stored direction = momentum * previous stored direction + current gradient`

`new parameter = old parameter - learning rate * stored direction`

On the first step, PyTorch initializes the stored direction from the first gradient.
There is no earlier history, so the two optimizers should make the same first update.
From the second step onward, momentum can change the direction used for updates.

Run sections 2 and 3 first. Compare momentum 0 and 0.9 with identical starting parameters,
learning rate 0.05, batch size 4, and 10 updates. Every printed row shows:

- `grad_w`: the weight gradient calculated **before** the update.
- `direction_w`: the direction actually used by SGD for that update, including momentum if enabled.
- `weight`, `bias`, and `loss`: values measured **after** the update.

`optimizer.zero_grad()` clears current gradients but preserves momentum state.
Create the optimizer once before the loop so its memory continues across steps.
Momentum can help maintain progress in a consistent direction, but can also carry
parameters past a good position. It does not guarantee lower loss within every budget.

### Experiment notes

- **Prediction:** the first update should match; later updates can differ because momentum remembers earlier gradients.
- **Observation:** _Compare steps 1, 2, and 3. In which run does loss rise after a good update?_
- **Explanation:** _Can the stored direction still be negative when the current gradient becomes positive? What would that do to the weight?_

Reference: [PyTorch SGD documentation](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html).
The experiment is verified against the installed PyTorch version.


In [ ]:
momentum_results = {}

for momentum in (0.0, 0.9):
    # Reset each run so momentum is the only changed setting.
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    optimizer = torch.optim.SGD(
        [w, b], lr=0.05, momentum=momentum, weight_decay=0
    )
    history = []
    print(f"\nMomentum = {momentum}")

    for step in range(1, 11):
        # Clear current gradients; retain the optimizer's momentum memory.
        optimizer.zero_grad(set_to_none=True)
        loss = ((w * x + b - y) ** 2).mean()
        loss.backward()
        current_gradient = w.grad.item()

        optimizer.step()

        # Inspect the direction used for this update; reading it does not change it.
        if momentum == 0:
            stored_direction = current_gradient
        else:
            stored_direction = optimizer.state[w]["momentum_buffer"].item()

        with torch.no_grad():
            updated_loss = ((w * x + b - y) ** 2).mean().item()

        history.append({
            "step": step,
            "gradient": current_gradient,
            "direction": stored_direction,
            "weight": w.item(),
            "bias": b.item(),
            "loss": updated_loss,
        })
        print(
            f"Step {step:2d} | grad_w={current_gradient:8.4f} | "
            f"direction_w={stored_direction:8.4f} | "
            f"weight={w.item():.4f} | bias={b.item():.4f} | loss={updated_loss:.4f}"
        )

    momentum_results[momentum] = history


## 25. Compare momentum settings at a fixed training budget

<a id="lesson-25"></a>

**Prediction before running:** momentum 0.5 should reduce overshooting and produce a
better result than 0.9 in this experiment. Test that expectation rather than treating
it as a guarantee.

Momentum controls how much of the previous stored direction survives. Learning rate
scales the resulting direction into a parameter change. A lower momentum coefficient
means less retained history; it does not guarantee a smaller update or lower final loss.
Current and previous gradients can reinforce or oppose each other.

Compare momentum 0, 0.5, and 0.9 from the same starting weight 1 and bias 0. Hold learning
rate 0.05, batch size 4, original targets, and a ten-update budget fixed. Run sections 2
and 3 first; this cell is independent of section 24's recorded results.

Inspect loss after **2 updates** and after **10 updates**. A setting can make faster
initial progress yet perform worse at a later budget. State the budget when saying
which setting performed better. These are training-loss comparisons; choosing settings
for unseen performance should use validation results.

### Experiment notes

- **Prediction:** momentum 0.5 will improve the ten-update result compared with 0.9.
- **Observation:** _Which setting has the lowest loss after two updates? Which after ten?_
- **Explanation:** _Why is 'lower momentum is always better' too broad a conclusion?_


In [ ]:
def run_momentum_experiment(momentum, steps=10):
    """Change only momentum; reset the parameters and optimizer for each run."""
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    optimizer = torch.optim.SGD(
        [w, b], lr=0.05, momentum=momentum, weight_decay=0
    )
    history = []

    for step in range(1, steps + 1):
        # Preserve learned parameters and momentum memory between steps.
        optimizer.zero_grad(set_to_none=True)
        loss = ((w * x + b - y) ** 2).mean()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            updated_loss = ((w * x + b - y) ** 2).mean().item()
        history.append({
            "step": step, "weight": w.item(),
            "bias": b.item(), "loss": updated_loss,
        })

    return history


# Identical data, initialization, learning rate, and update count in every run.
momentum_comparison = {}
for momentum in (0.0, 0.5, 0.9):
    history = run_momentum_experiment(momentum)
    momentum_comparison[momentum] = history
    print(f"\nMomentum = {momentum}")
    for step in (1, 2, 3, 5, 10):
        row = history[step - 1]
        print(
            f"Step {step:2d} | weight={row['weight']:.4f} | "
            f"bias={row['bias']:.4f} | loss={row['loss']:.4f}"
        )

print("\nSame settings, different update budgets:")
print("Momentum | Loss after 2 updates | Loss after 10 updates")
for momentum, history in momentum_comparison.items():
    print(f"{momentum:8.1f} | {history[1]['loss']:20.6f} | {history[9]['loss']:21.6f}")


## 26. Learning rate and momentum interact

<a id="lesson-26"></a>

Momentum combines previous and current gradients into a stored direction. Learning rate
scales that direction into a parameter update. For the same direction −29.5, learning
rate 0.05 gives an increase of 1.475, while rate 0.01 gives an increase of 0.295.

**Prediction:** lowering learning rate may reduce the large swings observed with momentum
0.9. It does not guarantee better performance at every training budget. The first step
will be smaller because both runs start at the same gradients; later gradients and stored
directions can differ as the runs follow different parameter trajectories.

Run sections 2 and 3 first. Keep momentum 0.9, original targets, batch size 4, starting
weight 1 and bias 0, and 100 updates fixed. Change only learning rate: 0.05 versus 0.01.
Evaluate the same reserved validation examples after each update without training on them.
Every printed loss is measured after that update using the same current parameters.

Compare the results at 10 and 100 updates. Training loss describes optimization progress;
validation loss helps assess performance on reserved examples and choose settings.

### Experiment notes

- **Prediction:** smaller initial updates and possibly smaller swings at learning rate 0.01.
- **Observation:** _Which rate has lower training and validation losses at 10 updates? At 100?_
- **Explanation:** _Why are the later updates not necessarily exactly five times smaller, even though the learning rates differ by a factor of five?_


In [ ]:
learning_rate_results = {}
validation_x = torch.tensor([4., 5.])
validation_y = torch.tensor([14., 17.])

for learning_rate in (0.05, 0.01):
    # Reset both parameters and the optimizer's history for an independent run.
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)
    optimizer = torch.optim.SGD(
        [w, b], lr=learning_rate, momentum=0.9, weight_decay=0
    )
    history = []
    print(f"\nLearning rate = {learning_rate}, momentum = 0.9")

    for step in range(1, 101):
        optimizer.zero_grad(set_to_none=True)
        loss = ((w * x + b - y) ** 2).mean()
        loss.backward()
        optimizer.step()

        # Measure both datasets without gradients or further parameter updates.
        with torch.no_grad():
            training_loss = ((w * x + b - y) ** 2).mean().item()
            validation_loss = (
                (w * validation_x + b - validation_y) ** 2
            ).mean().item()

        history.append({
            "step": step, "weight": w.item(), "bias": b.item(),
            "training_loss": training_loss, "validation_loss": validation_loss,
        })
        if step in (1, 2, 3, 5, 10, 20, 50, 100):
            print(
                f"Step {step:3d} | weight={w.item():.4f} | bias={b.item():.4f} | "
                f"training={training_loss:.6f} | validation={validation_loss:.6f}"
            )

    learning_rate_results[learning_rate] = history

print("\nSame momentum, different learning rates and update budgets:")
print("Rate | Train at 10 | Valid at 10 | Train at 100 | Valid at 100")
for rate, history in learning_rate_results.items():
    at_10 = history[9]
    at_100 = history[99]
    print(
        f"{rate:.2f} | {at_10['training_loss']:.6f} | {at_10['validation_loss']:.6f} | "
        f"{at_100['training_loss']:.6f} | {at_100['validation_loss']:.6f}"
    )


## 27. Select and load the best validation checkpoint

<a id="lesson-27"></a>

Select the saved parameters with the lowest validation loss, rather than automatically
using the final parameters. Run section 26 first; this cell reads its `learning_rate_results`
histories and does not perform any further training.

For each learning rate, find the lowest validation loss across the same 100-update budget.
Compare those two best checkpoints using validation loss, then load the selected weight
and bias into separate tensors for evaluation. These choices use validation data, not test data.

`min(history, key=lambda row: row["validation_loss"])` finds the recorded row with the
smallest validation loss. Its stored weight and bias are numeric snapshots from that
update. `.copy()` keeps a separate checkpoint record when adding configuration information.

The final row and best row can differ. Very small losses on this tiny validation set
are evidence about these two reserved examples; they do not guarantee results on all inputs.

### Experiment notes

- **Prediction:** the best checkpoint may occur before the last update.
- **Observation:** _For each rate, record the best step, best validation loss, and final validation loss._
- **Explanation:** _Why should evaluation use the saved parameters from that best step rather than the last global `w` and `b`?_


In [ ]:
rate_checkpoints = {}
print("Rate | Best step | Best validation | Final validation")

for rate, history in learning_rate_results.items():
    # Search all recorded updates using the validation-selection criterion.
    best = min(history, key=lambda row: row["validation_loss"])
    checkpoint = best.copy()
    checkpoint["learning_rate"] = rate
    checkpoint["momentum"] = 0.9
    rate_checkpoints[rate] = checkpoint
    print(
        f"{rate:.2f} | {best['step']:9d} | {best['validation_loss']:.8f} | "
        f"{history[-1]['validation_loss']:.8f}"
    )

# Choose among the best checkpoints, rather than comparing only final rows.
selected_rate = min(
    rate_checkpoints,
    key=lambda rate: rate_checkpoints[rate]["validation_loss"],
)
selected_checkpoint = rate_checkpoints[selected_rate].copy()

# Load the chosen saved numbers, preserving earlier runs and optimizer objects.
selected_w = torch.tensor(selected_checkpoint["weight"])
selected_b = torch.tensor(selected_checkpoint["bias"])
with torch.no_grad():
    selected_predictions = selected_w * validation_x + selected_b
    selected_validation_loss = (
        (selected_predictions - validation_y) ** 2
    ).mean().item()

print("\nSelected learning rate:", selected_rate)
print("Selected step:", selected_checkpoint["step"])
print("Selected weight and bias:", selected_w.item(), selected_b.item())
print("Validation predictions:", selected_predictions.tolist())
print(f"Validation loss after loading: {selected_validation_loss:.8f}")
